# Render MIDI with SINE Player VST

本 notebook 用于调用：

`D:\organ-amt-generalization\scripts\render_midi_sine.py`

将 MIDI 文件夹中的 `.mid / .midi` 批量渲染为单声道 `PCM_16` WAV。

输出目录固定为：

`D:\organ-amt-generalization\data\rendered\BWV 545`

运行前确认当前 notebook 内核为：

`Python (daw)`

In [ ]:
from pathlib import Path
import sys
import os
import platform

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Platform:", platform.platform())

## 1. 设置绝对路径

In [ ]:
from pathlib import Path

SCRIPT_PATH = Path(r"D:\organ-amt-generalization\scripts\render_midi_sine.py")

MIDI_DIR = Path(r"D:\organ-amt-generalization\data\raw\organ\ex01_bachorgan\midi")

SAVE_DIR = Path(r"D:\organ-amt-generalization\data\rendered\ttmp")

CONFIG_PATH = Path(r"D:\organ-amt-generalization\configs\render_sine.yaml")

print("SCRIPT_PATH:", SCRIPT_PATH)
print("SCRIPT exists:", SCRIPT_PATH.exists())

print("MIDI_DIR:", MIDI_DIR)
print("MIDI_DIR exists:", MIDI_DIR.exists())

print("SAVE_DIR:", SAVE_DIR)
print("SAVE_DIR exists:", SAVE_DIR.exists())

print("CONFIG_PATH:", CONFIG_PATH)
print("CONFIG exists:", CONFIG_PATH.exists())

SAVE_DIR.mkdir(parents=True, exist_ok=True)

## 2. 检查配置文件

In [ ]:
import yaml

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

print(cfg)

vst_path = Path(cfg["vst_path"])
print("VST path:", vst_path)
print("VST exists:", vst_path.exists())

if not vst_path.exists():
    raise FileNotFoundError(f"VST 插件不存在: {vst_path}")

# 如果配置里没有 wav_subtype，脚本会默认使用 PCM_16。
print("wav_subtype:", cfg.get("wav_subtype", "PCM_16"))

## 3. 检查 MIDI 输入文件夹

In [ ]:
midi_files = sorted(list(MIDI_DIR.glob("*.mid")) + list(MIDI_DIR.glob("*.midi")))

print("MIDI file count:", len(midi_files))

for p in midi_files:
    print(p)

if len(midi_files) == 0:
    raise FileNotFoundError(f"没有找到 MIDI 文件: {MIDI_DIR}")

## 4. 清理旧输出（可选）

默认不删除旧文件。若需要重新渲染，运行本 cell 删除旧的 WAV 和 metadata。

In [ ]:
# 可选：删除旧 wav 和旧 metadata
# 如不想删除，跳过本 cell。

DELETE_OLD_OUTPUTS = False

if DELETE_OLD_OUTPUTS:
    for p in SAVE_DIR.glob("*.wav"):
        print("delete:", p)
        p.unlink()

    meta = SAVE_DIR / "render_metadata.csv"
    if meta.exists():
        print("delete:", meta)
        meta.unlink()

print("current wav files:")
for p in sorted(SAVE_DIR.glob("*.wav")):
    print(p)

## 5. 用绝对路径加载渲染脚本

这里不使用 `import render_midi_sine`，而是直接按文件路径加载，避免 VS Code / notebook 无法解析模块路径。

In [ ]:
import importlib.util
import sys

spec = importlib.util.spec_from_file_location(
    "render_midi_sine_module",
    SCRIPT_PATH
)

render_midi_sine_module = importlib.util.module_from_spec(spec)
sys.modules["render_midi_sine_module"] = render_midi_sine_module
spec.loader.exec_module(render_midi_sine_module)

render_midi_to_wav = render_midi_sine_module.render_midi_to_wav

print("render_midi_sine.py loaded successfully.")
print("function:", render_midi_to_wav)

## 6. 开始渲染

第一次运行建议：

`OPEN_EDITOR_FIRST = True`

SINE Player 打开后：

1. 进入 Crucible；
2. 双击 `01 Organ`；
3. 确认右侧 `Loaded Articulations` 中有 Organ；
4. 关闭 SINE Player 窗口；
5. notebook 会继续渲染所有 MIDI。

后续如果插件能记住音色，可以改成：

`OPEN_EDITOR_FIRST = False`

In [ ]:
OPEN_EDITOR_FIRST = True
OVERWRITE = True

out_wavs = render_midi_to_wav(
    midi_path=MIDI_DIR,
    save_path=SAVE_DIR,
    config_path=CONFIG_PATH,
    open_editor_first=OPEN_EDITOR_FIRST,
    overwrite=OVERWRITE,
)

out_wavs

## 7. 检查输出 WAV 文件

In [ ]:
wav_files = sorted(SAVE_DIR.glob("*.wav"))

print("WAV file count:", len(wav_files))
for p in wav_files:
    print(p)

if len(wav_files) == 0:
    raise FileNotFoundError(f"没有生成 WAV 文件: {SAVE_DIR}")

## 8. 检查 WAV 是否为单声道、是否静音、是否 PCM_16

In [ ]:
import numpy as np
import pandas as pd
import soundfile as sf

rows = []

SEGMENT_START_SEC = 0.0
SEGMENT_DURATION_SEC = 30.0

for wav_path in wav_files:
    info = sf.info(str(wav_path))
    audio, sr = sf.read(str(wav_path), always_2d=True)

    duration_sec = len(audio) / sr

    peak_abs = float(np.max(np.abs(audio))) if audio.size > 0 else 0.0
    rms = float(np.sqrt(np.mean(audio ** 2))) if audio.size > 0 else 0.0
    nonzero_ratio = float(np.mean(np.abs(audio) > 1e-8)) if audio.size > 0 else 0.0

    start_sample = int(SEGMENT_START_SEC * sr)
    end_sample = int((SEGMENT_START_SEC + SEGMENT_DURATION_SEC) * sr)
    segment = audio[start_sample:end_sample]

    segment_peak_abs = float(np.max(np.abs(segment))) if segment.size > 0 else 0.0
    segment_rms = float(np.sqrt(np.mean(segment ** 2))) if segment.size > 0 else 0.0

    rows.append({
        "file": wav_path.name,
        "path": str(wav_path),
        "sample_rate": sr,
        "channels": audio.shape[1],
        "duration_sec": duration_sec,
        "format": info.format,
        "subtype": info.subtype,

        "peak_abs": peak_abs,
        "rms": rms,
        "nonzero_ratio": nonzero_ratio,
        "is_silent_full": peak_abs < 1e-8,

        "segment_start_sec": SEGMENT_START_SEC,
        "segment_duration_sec": SEGMENT_DURATION_SEC,
        "segment_peak_abs": segment_peak_abs,
        "segment_rms": segment_rms,
        "is_silent_segment": segment_peak_abs < 1e-8,
    })

df_wav_check = pd.DataFrame(rows)
display(df_wav_check)

print("Expected:")
print("- channels == 1")
print("- subtype == PCM_16")
print("- is_silent_full == False")

## 9. 读取 render_metadata.csv

In [ ]:
metadata_path = SAVE_DIR / "render_metadata.csv"

print("metadata_path:", metadata_path)
print("exists:", metadata_path.exists())

if metadata_path.exists():
    df_meta = pd.read_csv(metadata_path)
    display(df_meta)
else:
    print("没有找到 render_metadata.csv")

## 10. Notebook 内试听输出 WAV

In [ ]:
'''
from IPython.display import Audio, display

for wav_path in wav_files:
    print(wav_path.name)
    display(Audio(str(wav_path)))
    '''

In [ ]:
# =========================
# CQT 子图：每次重新读取当前 SAVE_DIR，固定画前 60 秒
# =========================

from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import librosa.display
import matplotlib.pyplot as plt

# 清理旧图，避免 notebook 显示旧 figure
plt.close("all")

# 必须使用前面已经定义好的 SAVE_DIR
if "SAVE_DIR" not in globals():
    raise NameError("当前 notebook 中没有 SAVE_DIR。请先运行路径配置单元格。")

SAVE_DIR = Path(SAVE_DIR)

print("Current SAVE_DIR:", SAVE_DIR)
print("SAVE_DIR exists:", SAVE_DIR.exists())

if not SAVE_DIR.exists():
    raise FileNotFoundError(f"SAVE_DIR 不存在: {SAVE_DIR}")

# 每次运行都重新读取 wav，不使用旧变量
wav_files = sorted(SAVE_DIR.glob("*.wav"))

print("WAV count:", len(wav_files))
print("WAV files used for plotting:")
for p in wav_files:
    print(" -", p)

if len(wav_files) == 0:
    raise FileNotFoundError(f"没有找到 wav 文件: {SAVE_DIR}")

# 固定画前 60 秒，不跳过静音
SEGMENT_START_SEC = 0.0
SEGMENT_DURATION_SEC = 180.0

# CQT 参数
TARGET_SR = 44100
CQT_HOP_LENGTH = 512
CQT_BINS_PER_OCTAVE = 36
CQT_N_BINS = 36 * 7
CQT_FMIN = librosa.note_to_hz("C1")

# 每次保存新图片，避免和旧图混淆
time_tag = datetime.now().strftime("%Y%m%d_%H%M%S")
folder_tag = SAVE_DIR.name.replace(" ", "_")

CQT_FIG_PATH = SAVE_DIR / f"{folder_tag}_cqt_first_60s_{time_tag}.png"

print("CQT_FIG_PATH:", CQT_FIG_PATH)


def load_audio_segment(audio_path, start_sec, duration_sec, target_sr=44100):
    y, sr = librosa.load(
        str(audio_path),
        sr=target_sr,
        mono=True,
        offset=start_sec,
        duration=duration_sec,
    )

    if len(y) == 0:
        raise ValueError(f"empty audio segment: {audio_path}")

    return y, sr


def compute_cqt_db(y, sr):
    cqt = librosa.cqt(
        y=y,
        sr=sr,
        hop_length=CQT_HOP_LENGTH,
        fmin=CQT_FMIN,
        n_bins=CQT_N_BINS,
        bins_per_octave=CQT_BINS_PER_OCTAVE,
    )

    mag = np.abs(cqt)

    if np.max(mag) <= 0:
        return np.full_like(mag, -80.0)

    return librosa.amplitude_to_db(mag, ref=np.max)


# 检查实际绘图片段信息
check_rows = []

for wav_path in wav_files:
    audio, sr = sf.read(wav_path, always_2d=True)

    start_sample = int(SEGMENT_START_SEC * sr)
    end_sample = int((SEGMENT_START_SEC + SEGMENT_DURATION_SEC) * sr)

    segment = audio[start_sample:end_sample]
    mono_segment = segment.mean(axis=1) if segment.ndim == 2 else segment

    peak_abs = float(np.max(np.abs(mono_segment))) if mono_segment.size > 0 else 0.0
    rms = float(np.sqrt(np.mean(mono_segment ** 2))) if mono_segment.size > 0 else 0.0

    check_rows.append({
        "file": wav_path.name,
        "path": str(wav_path),
        "modified_time": datetime.fromtimestamp(wav_path.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S"),
        "sample_rate": sr,
        "channels": audio.shape[1],
        "duration_sec": len(audio) / sr,
        "segment_start_sec": SEGMENT_START_SEC,
        "segment_duration_sec": SEGMENT_DURATION_SEC,
        "segment_peak_abs": peak_abs,
        "segment_rms": rms,
        "segment_is_silent": peak_abs < 1e-8,
    })

df_cqt_check = pd.DataFrame(check_rows)
display(df_cqt_check)


# 画 CQT 子图
n = len(wav_files)

fig, axes = plt.subplots(
    nrows=n,
    ncols=1,
    figsize=(16, 4.8 * n),
    constrained_layout=True,
)

if n == 1:
    axes = [axes]

last_img = None

for ax, wav_path in zip(axes, wav_files):
    y, sr = load_audio_segment(
        audio_path=wav_path,
        start_sec=SEGMENT_START_SEC,
        duration_sec=SEGMENT_DURATION_SEC,
        target_sr=TARGET_SR,
    )

    cqt_db = compute_cqt_db(y, sr)

    last_img = librosa.display.specshow(
        cqt_db,
        sr=TARGET_SR,
        hop_length=CQT_HOP_LENGTH,
        x_axis="time",
        y_axis="cqt_note",
        fmin=CQT_FMIN,
        bins_per_octave=CQT_BINS_PER_OCTAVE,
        ax=ax,
    )

    ax.set_title(
        f"{wav_path.name}\n"
        f"path={wav_path}\n"
        f"CQT from {SEGMENT_START_SEC:.1f}s to {SEGMENT_START_SEC + SEGMENT_DURATION_SEC:.1f}s"
    )

if last_img is not None:
    fig.colorbar(last_img, ax=axes, format="%+2.0f dB")

fig.suptitle(
    f"CQT first 60 seconds | SAVE_DIR = {SAVE_DIR}",
    fontsize=14,
)

fig.savefig(CQT_FIG_PATH, dpi=150)
plt.show()

print("saved:", CQT_FIG_PATH)